# Main S3a — reader, main conditions
**Run with: Save Version → "Save & Run All (Commit)"**, accelerator **GPU T4 x2**, Internet On. It runs unattended; if any step fails the notebook stops with `FAILED` — send me the log. When it finishes, download `main_s3a_results.zip` from the version's Output tab and send it to me.

**Input:** Add Input → Your Work → the S1 notebook (its saved output with `main_s1_results.zip`).

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 1 — hardware, code frozen at 22f161a (branch main-v1), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
sh("git clone --depth 1 --branch main-v1 https://github.com/adeelcheena791/rag-embed-transfer.git rag")
os.chdir("/kaggle/working/rag")
sh("git log --oneline -1")
sh("test \"$(git rev-parse HEAD)\" = 22f161a6e48a188ce8114bc8f6d402bcf7fdf327 || (echo WRONG CODE COMMIT; exit 1)")
sh("pip install -q -r requirements-retrieval-pinned.txt")
sh("pip install -q -r requirements-generation-pinned.txt")
sh("python -m src.check_env --lock requirements-generation.lock")

sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore S1 outputs (input: the S1 notebook's output) FIRST (nothing may write results/ before this), then environment record + corpus text
sh("python -m src.restore --name main_s1_results")
sh("python -m src.env_report --session main_S3a --out results/raw/env_main_S3a.json")
sh("python -m src.data.download --config configs/main.yaml")

In [ ]:
# Cell 3 — main reader run: 9 conditions x 2,255 queries = 20,295 generations (~3.3 h)
sh("CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/main.yaml --conditions R0,R1,R2,R3,R4,R5,R6,CB,GOLD --corpus main --n 2255 --k 5 --auto-eager-retry")

In [ ]:
# Cell 4 — score answers (processed files; the analysis itself happens offline)
sh("python -m src.evaluation.answer --config configs/main.yaml --prefix main")

In [ ]:
# Last cell — summary + package (becomes this version's saved output)
sh("python -m src.pilot_summary --config configs/main.yaml")
sh("zip -qr /kaggle/working/main_s3a_results.zip results data/processed configs/resolved_revisions.json logs requirements-*.lock data/raw/nq_qrels/test.tsv data/raw/nq_corpus/queries")
sh("ls -la /kaggle/working/main_s3a_results.zip")
sh("rm -rf /kaggle/working/rag/data/raw /tmp/models")   # re-downloadable; keeps the saved output small